In [1]:
import os
import sys
import torch
import numpy as np
from PIL import Image

# ── Paths ──
BASE_DIR = "/home/rishabh/Downloads/umi-pipeline-training"
RDT2_DIR = f"{BASE_DIR}/RDT2"
CHECKPOINT_DIR = f"{BASE_DIR}/outputs/rdt2-finetuned/checkpoint-5000"
NORMALIZER_PATH = f"{BASE_DIR}/umi_normalizer_official.pt"

sys.path.insert(0, RDT2_DIR)
print("✅ Paths initialized.")

✅ Paths initialized.


In [2]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel
from models.normalizer.normalizer import LinearNormalizer
from vqvae.models.multivqvae import MultiVQVAE

device = torch.device("cuda")
dtype = torch.bfloat16

# 1. Load Processor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", padding_side="left")

# 2. Load VLM + Merge LoRA (Removes 12-second lag)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",
    torch_dtype=dtype, device_map={"": device}, attn_implementation="flash_attention_2"
)
model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
model = model.merge_and_unload() # ✅ Merging for speed
model.visual.to(dtype=dtype)     # ✅ Vision speed fix
model.eval()

# 3. Load VAE (Matched to your 30-token training)
vae = MultiVQVAE.from_pretrained(
    "robotics-diffusion-transformer/RVQActionTokenizer",
    n_codebooks={"pos": 6, "rot": 3, "grip": 1} 
)
vae.to(device=device, dtype=torch.float32).eval()
vae.pos_id_len, vae.rot_id_len, vae.grip_id_len = 18, 9, 3

# 4. Load Normalizer
normalizer = LinearNormalizer.load(NORMALIZER_PATH)
normalizer.to(device)

print("✅ ALL MODELS LOADED & OPTIMIZED")

/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 4/4 [00:09<00:00,  2.43s/it]


LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
✅ ALL MODELS LOADED & OPTIMIZED


In [3]:
import torch.nn.functional as F

def rot6d_to_axis_angle(rot6d):
    x_raw, y_raw = rot6d[..., 0:3], rot6d[..., 3:6]
    b1 = F.normalize(x_raw, dim=-1)
    b2 = F.normalize(y_raw - torch.sum(b1 * y_raw, dim=-1, keepdim=True) * b1, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    # Extract matrix components
    r11, r22, r33 = b1[..., 0], b2[..., 1], b3[..., 2]
    angle = torch.acos(torch.clamp((r11 + r22 + r33 - 1.0) / 2.0, -1.0, 1.0))
    scale = torch.where(angle > 1e-4, angle / (2.0 * torch.sin(angle) + 1e-9), 0.5)
    return torch.stack([(b3[..., 1]-b2[..., 2])*scale, (b1[..., 2]-b3[..., 0])*scale, (b2[..., 0]-b1[..., 1])*scale], dim=-1)

def predict_robot_thought(image, instruction):
    inputs = processor(text=[processor.apply_chat_template([{"role":"user","content":[{"type":"image"},{"type":"text","text":instruction}]}], add_generation_prompt=True)],
                       images=[[image]], padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        action_tokens = output_ids[0, inputs.input_ids.shape[1]:].clamp(0, 1023).long()
        # Decode VAE (20-D Bimanual)
        raw_vae = vae.decode(action_tokens[:30].unsqueeze(0))
        # Bridge to 7-D (Arm 0)
        pos, rot_6d, grip = raw_vae[..., 0:3], raw_vae[..., 3:9], raw_vae[..., 9:10]
        combined = torch.cat([pos, rot6d_to_axis_angle(rot_6d), grip], dim=-1)
        # Unnormalize
        return normalizer["action"].unnormalize(combined).detach().cpu().numpy()[0]

print("✅ Mathematical Bridge Defined.")

✅ Mathematical Bridge Defined.


In [4]:
def get_m750_safe_commands(actions):
    Y_SHIFT = 150.0 # Moves UMI world into M750 reach
    commands = []
    for a in actions:
        pos_mm = [a[0]*1000, (a[1]*1000) + Y_SHIFT, a[2]*1000]
        # Hard Physical Clamp for M750 safety
        pos_mm = [np.clip(pos_mm[0], -300, 300), np.clip(pos_mm[1], -305, 305), np.clip(pos_mm[2], 20, 350)]
        commands.append({
            'coords': pos_mm + np.rad2deg(a[3:6]).tolist(),
            'gripper': 0 if a[6] > 0.5 else 1 # 0=Open, 1=Closed
        })
    return commands

print("✅ M750 Command Processor Ready.")

✅ M750 Command Processor Ready.


In [6]:
from pymycobot.myarm import MyArm
import time

# 1. Hardware Connect
PORT = "/dev/ttyACM0" # Ensure this is correct!
mc = MyArm(PORT, 115200)
mc.power_on()
time.sleep(1)

# 2. Provide a test image (Neutral gray or your own 'test.jpg')
test_img = Image.new('RGB', (384, 384), color=(128, 128, 128))

print("🧠 AI is thinking...")
actions = predict_robot_thought(test_img, "pick up the marker and place it in the box")
m750_setpoints = get_m750_safe_commands(actions)

# 3. Take the first move
target = m750_setpoints[0]

print(f"🚀 PHYSICALLY MOVING ROBOT TO: {np.round(target['coords'][:3], 1)}")

# 4. SEND TO MOTORS
# Speed 20 (Safe), Mode 1 (Linear Move)
mc.send_coords(target['coords'], 20, 1)
mc.set_gripper_state(target['gripper'], 50)

print("✅ Robot should have moved to the starting position.")

🧠 AI is thinking...


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


🚀 PHYSICALLY MOVING ROBOT TO: [-174.7 -264.   147.5]
✅ Robot should have moved to the starting position.


In [ ]:
from pymycobot.myarm import MyArm
import time
import numpy as np

# 1. HARDWARE SETTINGS
# Change to "/dev/ttyACM1" if ACM0 doesn't work
PORT = "/dev/ttyACM0" 
BAUD = 115200 # ⚠️ Changed from 115200 to 1,000,000 for M750

print(f"🔄 Attempting high-speed connection on {PORT}...")
mc = MyArm(PORT, BAUD)
time.sleep(1)

# 2. WAKE UP SEQUENCE
print("⚡ Engaging motors and resetting mode...")
mc.power_on()
time.sleep(0.5)
# Fresh Mode 0: Standard execution
# Fresh Mode 1: Real-time (discard old commands)
mc.set_fresh_mode(0) 
time.sleep(1)

# 3. CONNECTION CHECK
angles = mc.get_angles()
if not angles or angles == -1:
    print("❌ ROBOT STILL NOT RESPONDING.")
    print("Action: Change BAUD back to 115200 in this cell and run again.")
else:
    print(f"✅ CONNECTION VERIFIED. Current Angles: {angles}")

    # 4. SEND THE PREDICTED MOVE
    # These are the numbers from your successful AI output
    target_xyz = [-174.7, -264.0, 147.5]
    # Standard orientation for 'pick' (facing down)
    # Adjust these based on your specific rotation output if needed
    target_rot = [1.79, 9.21, 10.7] 
    
    full_coords = target_xyz + target_rot
    https://file+.vscode-resource.vscode-cdn.net/dev/ttyACM0...
    print(f"🚀 PHYSICALLY MOVING TO: {target_xyz}")
    
    # Speed 20, Mode 1 (Linear Move)
    mc.send_coords(full_coords, 20, 1)
    
    # 5. GRIPPER
    # 1 is Close, 0 is Open for standard M750
    mc.set_gripper_state(1, 50) 
    
    time.sleep(2)
    print(f"📍 New Robot Position: {mc.get_coords()}")

🔄 Attempting high-speed connection on /dev/ttyACM0...
⚡ Engaging motors and resetting mode...
❌ ROBOT STILL NOT RESPONDING.
Action: Change BAUD back to 115200 in this cell and run again.


: 

In [ ]:
"""
STEP 2: Train VQ-VAE Tokenizer for D=7 (M750 + UMI gripper)
=============================================================
Run from RDT2 directory:
    cd /home/rishabh/Downloads/umi-pipeline-training/RDT2
    source /home/rishabh/Downloads/umi-pipeline-training/umi_env/bin/activate
    python step2_train_vqvae.py
"""

import os, sys, glob, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── CONFIG ────────────────────────────────────────────────────────────────────
SHARDS_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/shards"
OUTPUT_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/vqvae-m750-7dof"
RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
ACTION_DIM   = 7        # [x, y, z, rx, ry, rz, gripper]
ACTION_HZ    = 24       # action chunk size
BATCH_SIZE   = 32
EPOCHS       = 100
LR           = 1e-4
DEVICE       = "cuda:0" if torch.cuda.is_available() else "cpu"

sys.path.insert(0, RDT2_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Shards: {SHARDS_DIR}")
print(f"Output: {OUTPUT_DIR}")

# ── PATCH multivqvae.py FOR D=7 ───────────────────────────────────────────────
print("\n[1/4] Patching multivqvae.py for D=7...")

VQVAE_FILE = f"{RDT2_DIR}/vqvae/models/multivqvae.py"

# Backup original
import shutil
if not os.path.exists(VQVAE_FILE + ".bak_original"):
    shutil.copy(VQVAE_FILE, VQVAE_FILE + ".bak_original")
    print(f"  Backed up to {VQVAE_FILE}.bak_original")

D7_CODE = '''"""
multivqvae.py — PATCHED FOR M750 SINGLE ARM D=7
D=7 layout: [x, y, z, rx, ry, rz, gripper]
              0  1  2   3   4   5      6
"""
from typing import Union
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin
from vqvae.models.vqvae import VQVAE


def select_act_dim(x, act_type):
    """Extract sub-dims from D=7 tensor."""
    if act_type == "pos":   return x[..., 0:3]   # xyz
    elif act_type == "rot": return x[..., 3:6]   # rx,ry,rz
    elif act_type == "grip":return x[..., 6:7]   # gripper
    raise ValueError(f"Unknown: {act_type}")


def apply_act_dim(x, y, act_type):
    """Write sub-dims back into D=7 tensor."""
    if act_type == "pos":   x[..., 0:3] = y
    elif act_type == "rot": x[..., 3:6] = y
    elif act_type == "grip":x[..., 6:7] = y
    return x


class MultiVQVAE(nn.Module, PyTorchModelHubMixin):
    def __init__(self, input_dim, embedding_dim, cnn_config,
                 num_embeddings, action_horizon,
                 n_codebooks=[6,2,1],
                 codebook_restart_interval=64,
                 commitment_cost=0.25,
                 codebook_cost=0., local_rank=0):
        super().__init__()

        self.pos_vqvae = VQVAE(
            input_dim=input_dim["pos"],        # 3
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["pos"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.pos_id_len = n_codebooks["pos"] * 3

        self.rot_vqvae = VQVAE(
            input_dim=input_dim["rot"],        # 3
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["rot"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.rot_id_len = n_codebooks["rot"] * 3

        self.grip_vqvae = VQVAE(
            input_dim=input_dim["grip"],       # 1
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["grip"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.grip_id_len = n_codebooks["grip"] * 3

        self.action_dim     = input_dim["pos"] + input_dim["rot"] + input_dim["grip"]  # = 7
        self.action_horizon = action_horizon
        self.num_embeddings = num_embeddings

    def encode(self, x):
        if isinstance(x, dict):
            p, r, g = x["pos"], x["rot"], x["grip"]
        else:
            p = select_act_dim(x, "pos")
            r = select_act_dim(x, "rot")
            g = select_act_dim(x, "grip")
        return torch.cat([
            self.pos_vqvae.encode(p),
            self.rot_vqvae.encode(r),
            self.grip_vqvae.encode(g)], dim=-1)

    def decode(self, ids, return_dict=False):
        p_ids = ids[..., :self.pos_id_len]
        r_ids = ids[..., self.pos_id_len:self.pos_id_len+self.rot_id_len]
        g_ids = ids[..., self.pos_id_len+self.rot_id_len:]
        p = self.pos_vqvae.decode(p_ids)
        r = self.rot_vqvae.decode(r_ids)
        g = self.grip_vqvae.decode(g_ids)
        if return_dict:
            return {"pos": p, "rot": r, "grip": g}
        x = torch.zeros(ids.shape[0], self.action_horizon,
                        self.action_dim, device=ids.device)
        x = apply_act_dim(x, p, "pos")
        x = apply_act_dim(x, r, "rot")
        x = apply_act_dim(x, g, "grip")
        return x   # (B, T, 7)

    def calculate_loss(self, x, x_recon):
        return {
            "pos":  self.pos_vqvae.calculate_loss(
                        select_act_dim(x,"pos"),
                        select_act_dim(x_recon,"pos"), act_type="pos"),
            "rot":  self.rot_vqvae.calculate_loss(
                        select_act_dim(x,"rot"),
                        select_act_dim(x_recon,"rot"), act_type="rot"),
            "grip": self.grip_vqvae.calculate_loss(
                        select_act_dim(x,"grip"),
                        select_act_dim(x_recon,"grip"), act_type="grip"),
        }
'''

with open(VQVAE_FILE, "w") as f:
    f.write(D7_CODE)
print("  ✅ multivqvae.py patched for D=7")

# ── DATASET ───────────────────────────────────────────────────────────────────
print("\n[2/4] Loading M750 shard dataset...")

class M750Dataset(Dataset):
    def __init__(self, shards_dir, action_horizon=24):
        self.chunks = []
        files = sorted(glob.glob(f"{shards_dir}/**/*.pt", recursive=True))
        if not files:
            files = sorted(glob.glob(f"{shards_dir}/*.pt"))
        print(f"  Found {len(files)} shard files")

        for f in files:
            try:
                data = torch.load(f, map_location="cpu")
                # Handle different shard formats
                if isinstance(data, dict):
                    if "action" in data:
                        actions = data["action"]
                    elif "actions" in data:
                        actions = data["actions"]
                    else:
                        print(f"  Skip {f}: no 'action' key, keys={list(data.keys())}")
                        continue
                elif isinstance(data, torch.Tensor):
                    actions = data
                else:
                    continue

                actions = actions.float()
                if actions.ndim == 1:
                    actions = actions.unsqueeze(0)

                # Verify D=7
                if actions.shape[-1] != 7:
                    print(f"  Skip {f}: action dim={actions.shape[-1]} (need 7)")
                    continue

                # Sliding window chunks
                T = len(actions)
                stride = action_horizon // 2
                for i in range(0, T - action_horizon + 1, stride):
                    chunk = actions[i:i+action_horizon]
                    if len(chunk) == action_horizon:
                        self.chunks.append(chunk)
            except Exception as e:
                print(f"  Skip {f}: {e}")

        print(f"  Total chunks: {len(self.chunks)}")
        if len(self.chunks) == 0:
            raise RuntimeError(f"No D=7 data found in {shards_dir}!")

        # Print action stats
        all_actions = torch.stack(self.chunks).reshape(-1, 7)
        print(f"  Action min: {all_actions.min(0).values.tolist()}")
        print(f"  Action max: {all_actions.max(0).values.tolist()}")
        print(f"  Action mean:{all_actions.mean(0).tolist()}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]  # (T, 7)

dataset = M750Dataset(SHARDS_DIR, action_horizon=ACTION_HZ)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                     shuffle=True, num_workers=2, pin_memory=True)

# ── BUILD VQVAE ───────────────────────────────────────────────────────────────
print("\n[3/4] Building VQ-VAE for D=7...")

from vqvae.models.multivqvae import MultiVQVAE

VQVAE_CONFIG = {
    "input_dim":    {"pos": 3, "rot": 3, "grip": 1},
    "embedding_dim": 64,
    "cnn_config": {
        "channels": [64, 128, 256],
        "kernels":  [4, 4, 4],
        "strides":  [2, 2, 2],
    },
    "num_embeddings":  512,
    "action_horizon":  ACTION_HZ,
    "n_codebooks":  {"pos": 6, "rot": 2, "grip": 1},
}

vqvae = MultiVQVAE(**VQVAE_CONFIG).to(DEVICE)
print(f"  action_dim     = {vqvae.action_dim}")    # should be 7
print(f"  pos_id_len     = {vqvae.pos_id_len}")    # 18
print(f"  rot_id_len     = {vqvae.rot_id_len}")    # 6
print(f"  grip_id_len    = {vqvae.grip_id_len}")   # 3
print(f"  valid_id_len   = {vqvae.pos_id_len + vqvae.rot_id_len + vqvae.grip_id_len}")  # 27

# ── TRAIN ─────────────────────────────────────────────────────────────────────
print(f"\n[4/4] Training VQ-VAE for {EPOCHS} epochs...")

optimizer = torch.optim.Adam(vqvae.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5)

best_loss = float("inf")

for epoch in range(EPOCHS):
    vqvae.train()
    total = {"pos": 0., "rot": 0., "grip": 0., "total": 0.}
    n = 0

    for batch in loader:
        batch = batch.to(DEVICE)          # (B, T, 7)
        ids   = vqvae.encode(batch)       # (B, L)
        recon = vqvae.decode(ids)         # (B, T, 7)
        losses = vqvae.calculate_loss(batch, recon)

        loss = sum(v["loss"] for v in losses.values())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vqvae.parameters(), 1.0)
        optimizer.step()

        total["total"] += loss.item()
        for k in ["pos", "rot", "grip"]:
            total[k] += losses[k]["loss"].item()
        n += 1

    scheduler.step()
    avg = {k: v/n for k, v in total.items()}

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:>3}/{EPOCHS} | "
              f"total={avg['total']:.4f} | "
              f"pos={avg['pos']:.4f} | "
              f"rot={avg['rot']:.4f} | "
              f"grip={avg['grip']:.4f}")

    # Save checkpoint every 25 epochs
    if (epoch + 1) % 25 == 0:
        ckpt = f"{OUTPUT_DIR}/vqvae_epoch{epoch+1}.pt"
        torch.save({
            "epoch": epoch+1,
            "model": vqvae.state_dict(),
            "config": VQVAE_CONFIG,
            "loss": avg["total"],
        }, ckpt)
        print(f"  💾 Saved: {ckpt}")

    # Save best
    if avg["total"] < best_loss:
        best_loss = avg["total"]
        torch.save({
            "epoch": epoch+1,
            "model": vqvae.state_dict(),
            "config": VQVAE_CONFIG,
            "loss": best_loss,
        }, f"{OUTPUT_DIR}/vqvae_best.pt")

# Final save
torch.save({
    "epoch": EPOCHS,
    "model": vqvae.state_dict(),
    "config": VQVAE_CONFIG,
    "loss": avg["total"],
}, f"{OUTPUT_DIR}/vqvae_final.pt")

print(f"\n✅ VQ-VAE training complete!")
print(f"   Best loss : {best_loss:.4f}")
print(f"   Saved to  : {OUTPUT_DIR}/vqvae_final.pt")
print(f"\nNext: Run step3_build_normalizer.py")